# Rainfall Analysis and Planting Advisory

**Project:** PLP Data Analytics Assignment 2  
**Purpose:** Clean the Machakos station readings, analyse rainfall across five counties with NumPy, and prepare a short planting advisory.

The dataset contains monthly rainfall in millimetres from January to December. Rows represent counties and columns represent months.


## 1. Import libraries and define the dataset

In [1]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]


## 2. Clean the Machakos readings

The Machakos readings were supplied as text. The function below converts valid entries to integers. It catches the specific `ValueError` raised when a value cannot be converted, then replaces invalid entries with the rounded mean of the valid readings.

In [2]:
def clean_machakos(raw_values, month_names):
    # First collect valid readings so the imputation average excludes invalid entries.
    valid_values = []
    for value in raw_values:
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            continue

    imputed_value = round(sum(valid_values) / len(valid_values))

    cleaned_values = []
    imputed_months = []
    for month, value in zip(month_names, raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(imputed_value)
            imputed_months.append(month)

    return cleaned_values, imputed_months, imputed_value


machakos_clean, imputed_months, imputed_value = clean_machakos(machakos_raw, months)

print("Cleaned Machakos readings:", machakos_clean)
print("Valid-reading average rounded for imputation:", imputed_value, "mm")
print("Months imputed:", ", ".join(imputed_months))


Cleaned Machakos readings: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Valid-reading average rounded for imputation: 68 mm
Months imputed: April, September


## 3. Combine counties into a NumPy array

In [3]:
# Add the cleaned Machakos row to the four already-clean county rows.
R = np.array(clean_rows + [machakos_clean], dtype=int)

print("Combined rainfall array (mm):")
print(R)
print("Array shape:", R.shape)
print("Array dtype:", R.dtype)


Combined rainfall array (mm):
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array dtype: int64


## 4. Annual totals and monthly averages

In [4]:
# Sum across columns (axis=1) to get one annual total per county.
annual_totals = R.sum(axis=1)

# Average across rows (axis=0) to get one average for each month.
monthly_averages = R.mean(axis=0)

print("Annual rainfall totals (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\\nAverage rainfall per month across all five counties (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")


Annual rainfall totals (mm):
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm
\nAverage rainfall per month across all five counties (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## 5. Wettest month in each county

In [5]:
# argmax(axis=1) returns the column index of the highest monthly value in each row.
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_indices]

print("Wettest month by county:")
for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")


Wettest month by county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## 6. Count dry months

In [6]:
# A dry month is defined in this assignment as rainfall below 50 mm.
dry_month_mask = R < 50
dry_month_counts = dry_month_mask.sum(axis=1)

print("Number of dry months (below 50 mm):")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")


Number of dry months (below 50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## 7. Recommended two-month planting windows

For each county, add each month to the month immediately after it. `R[:, :-1] + R[:, 1:]` creates all eleven consecutive two-month rainfall totals for every county. `argmax(axis=1)` identifies the pair with the highest combined rainfall.

In [7]:
two_month_totals = R[:, :-1] + R[:, 1:]
planting_start_indices = np.argmax(two_month_totals, axis=1)
planting_windows = [
    f"{months[index]}–{months[index + 1]}"
    for index in planting_start_indices
]

print("Recommended two-month planting windows:")
for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")


Recommended two-month planting windows:
Nairobi: April–May
Nakuru: April–May
Kisumu: April–May
Mombasa: April–May
Machakos: November–December


## 8. Rainfall summary CSV

In [8]:
# Write exactly one data row per county, plus the CSV header.
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "county", "annual_total_mm", "wettest_month",
        "dry_months", "planting_window"
    ])

    for row in zip(counties, annual_totals, wettest_months,
                   dry_month_counts, planting_windows):
        writer.writerow([
            row[0], int(row[1]), row[2], int(row[3]), row[4]
        ])

print("rainfall_summary.csv has been created.")


rainfall_summary.csv has been created.


## Planting Advisory

The analysis suggests using each county's two consecutive months with the highest combined rainfall as an initial planting window: Nairobi (April–May), Nakuru (April–May), Kisumu (April–May), Mombasa (April–May), and Machakos (November–December). These windows identify the strongest two-month rainfall periods in the supplied annual dataset, so farmers should prepare land and seed ahead of the recommended period. The NGO should advise farmers to confirm the timing with current weather forecasts and local agricultural extension officers before planting. Farmers should also consider soil drainage, crop type, water availability and the risk of unusually heavy rainfall. The recommendations are based on one set of monthly readings and should not be treated as a guarantee of future rainfall.


## Data Quality Notes

The Machakos station data contained two invalid text entries: `N/A` for April and an empty string for September. Both were replaced with 68 mm, calculated by rounding the mean of the ten valid Machakos readings (67.5 mm). The other four county rows were used as supplied and were not altered. Imputation keeps the array complete for NumPy analysis, but it introduces estimated values that may affect Machakos totals, averages and its recommended window. The dataset covers only the twelve months shown, so the results describe this supplied dataset rather than long-term rainfall patterns; the NGO should validate station records and compare the advisory with updated forecasts.
